In [ ]:
import time, os, json, subprocess, sys, glob
T0=time.time()


In [ ]:
#@title Install dependencies (~90 s)
import os, time, glob, shutil, sys

model = "openbind0" #@param ["openbind0", "openfold3", "boltz2", "protenix2", "rosettafold3", "chai1", "intellifold2", "opendde", "esmfold2", "esmfold2_lm600m", "esmfold2_lm300m", "alphafold3", "af2_ptm", "af2_multimer"]
#@markdown - **model**: which set of weights to run. All of them use the same AlphaFold 3
#@markdown   graph, so everything downstream is identical. `openbind0` is OpenFold3's current
#@markdown   release and a good default; `openfold3` is their earlier preview-2, kept because
#@markdown   earlier results used it. The three `esmfold2*` entries fold from ESM-C instead
#@markdown   of an MSA -- single sequence, no search -- and differ only in the size of that
#@markdown   language model (6B, 600M, 300M). `chai1` and `esmfold2*` download and run their
#@markdown   language model automatically. `alphafold3` fetches Google DeepMind's own
#@markdown   parameters and is subject to the AF3 terms of use.

persist_cache_to_drive = False #@param {type:"boolean"}
#@markdown - **persist_cache_to_drive**: keep the compiled model in your Google Drive so
#@markdown   the next session does not recompile. Measured on a 68-residue input: the first
#@markdown   prediction takes **69 s** with a cold cache and **16 s** with a warm one, so this
#@markdown   is worth about **53 s per session** (more for longer inputs). Colab wipes `/tmp`
#@markdown   between sessions, which is why it has to go somewhere else to survive. Leaving
#@markdown   it off costs only that recompile; it never changes a result.

# PINNED, both halves. Until 2026-09-16 this installed the v3.1.5 wheel for its
# compiled extension and then overlaid the Python half from the BRANCH HEAD --
# so the notebook mixed a fixed binary with a moving source tree, and two runs
# on different days could be different code. `alphafold3-colabfold` is published
# on PyPI (cp312/cp313/cp314 manylinux + macOS arm64) and its Python half knows
# every model, so the overlay is gone and both the package and run_alphafold.py
# come from one tag.
VERSION = '3.1.8'
NATIVE_DIR = 'af3_native_weights'
AF3_WEIGHTS_URL = 'https://storage.googleapis.com/alphafold3/af3.bin.zst'
IS_AF3 = (model == 'alphafold3')
# AlphaFold 2 is a SIBLING NETWORK, not one of the AF3-family ports: MSA row and
# column attention into an IPA head, reached through the same CLI and writing the
# same outputs. Its parameters are DeepMind's own release under CC BY 4.0, so they
# are fetched from source. Protein only -- a ligand or nucleotide in the input
# raises rather than folding the protein part and saying nothing.
IS_AF2 = model.startswith('af2_')
AF2_DIR = 'af2_params'
# int8 everywhere: same weights stored 8-bit and expanded on load, which is
# what keeps a Colab download to a few hundred MB. Not a knob -- there is no
# reason to pick anything else here, and AlphaFold 3's own parameters come
# from Google as float32 regardless.
PRECISION = 'fp32' if (IS_AF3 or IS_AF2) else 'int8'

if not os.path.isfile('ALPHAFOLD3_READY'):
  print('Installing packages...')
  # THE SLIM WHEEL, from PyPI. It is 9 MB. Until v3.1.8 this had to be the
  # 130 MB `+data` wheel from a GitHub release, because importing
  # `alphafold3.cpp` died with
  #     ImportError: Could not find the libcifpp components.cif file.
  # unless the 518 MB dictionary was bundled. That error was OURS, not
  # libcifpp's: mkdssp_pybind.cc threw from module REGISTRATION, so the whole
  # extension -- including `cif_dict`, which featurisation needs -- failed to
  # import for the sake of DSSP, which no fold calls. v3.1.8 defers the check
  # to `get_dssp` itself. VERIFIED on the published 3.1.8 wheel in a clean
  # venv with no components.cif anywhere: the extension imports, a
  # protein+ligand input featurises, and get_dssp raises something actionable.
  # ml_collections and dm-tree ARE declared dependencies of the package, but it
  # goes in with --no-deps (so pip does not re-resolve jax and the CUDA stack
  # Colab already has), so every third-party import has to be listed here. Those two are imported ONLY
  # by the af2 path (`af2/model/config.py`, and dm-tree in three more), which is
  # why every af3-family model worked and `--model af2_ptm` died with
  # `ModuleNotFoundError: No module named 'ml_collections'`.
  # The full set under src/alphafold3/af2 is: absl, haiku, jax, ml_collections,
  # numpy, scipy, tree -- the rest are already here or in Colab's base image.
  # MEASURED against a fresh Colab image (2026-09-16, py 3.13.15, T4), not
  # guessed. Already present, so not installed: zstandard 0.25.0, dm-tree
  # 0.1.10, numpy 2.1.3, scipy 1.16.3, absl-py, and 21 nvidia CUDA wheels.
  # Absent, so installed: dm-haiku, rdkit, tokamax, ml_collections, py2Dmol.
  #
  # DROPPED: awscli and py3Dmol were installed and never used -- `aws` is never
  # invoked and only py2Dmol is imported. awscli alone drags in the boto stack.
  # NO JAX PIN. This used to force jax[cuda12]==0.10.1; Colab ships 0.11.1, so
  # the pin downgraded jax AND re-pulled the whole CUDA wheel stack -- the
  # dominant cost of this cell. Measured on a fresh T4 session: these four
  # install in 8.5 s against minutes with the pin, and jax 0.11.1 imports,
  # traces and folds correctly (openbind0, 20 residues, rc=0, 165 atoms).
  #
  # CAVEAT, stated because it is untested rather than dismissed: that check was
  # on a T4, which takes the XLA attention path. tokamax's Triton kernels are
  # restricted to datacenter GPUs below, so an A100/H100 run exercises code a
  # T4 does not. If a datacenter GPU misbehaves, pin jax again here first.
  os.system("pip install -q dm-haiku==0.0.17 rdkit==2025.9.4 \
  tokamax==0.0.11 ml_collections")
  # py2Dmol from source: the released wheel lags the repo.
  os.system("pip install -q git+https://github.com/sokrypton/py2Dmol.git")
  # aria2c, for AF2 only: its parameter tar is 5.3 GB and a single connection is
  # the bottleneck, not the link (weights._download_parallel uses it when it is
  # on PATH). Every af3-family blob is 130-350 MB, where this would not pay.
  if IS_AF2:
    os.system("apt-get -qq install -y aria2 > /dev/null 2>&1")
  # --no-deps: the package declares jax, and Colab already ships a working
  # one -- resolving its dependencies would re-pull the whole CUDA wheel
  # stack, which is why every third-party import is listed above instead.
  os.system(f'pip install -q --no-deps alphafold3-colabfold=={VERSION}')
  # `run_alphafold.py` is a top-level script, not part of the package
  # (`wheel.packages = ["src/alphafold3"]`), so the wheel does not carry it.
  # Fetch it AT THE TAG so the driver and the library are the same commit.
  os.system(f'wget -q -O run_alphafold.py https://raw.githubusercontent.com'
            f'/sokrypton/alphafold3/v{VERSION}/run_alphafold.py')
  # haiku 0.0.17 still calls the moved `jax.core.DropVar`; checked against the
  # installed 0.0.17 tree, this one is still needed. (A second sed for
  # `jax.core.get_opaque_trace_state` used to sit here and never matched --
  # base.py reaches it through a `jax_core` alias and already falls back to
  # `jex_core` itself, so it was only ever a no-op.)
  os.system("sed -i 's/jax.core.DropVar/jax.extend.core.DropVar/g' /usr/local/lib/python*/dist-packages/haiku/_src/jaxpr_info.py")
  os.system('touch ALPHAFOLD3_READY')
  print('Packages installed.')

# Patch tokamax so Ada/consumer GPUs (L4, A10, RTX 30/40; cc 8.6/8.9) fall back to XLA
# kernels. tokamax enables its Triton kernels for ALL cc>=8.0 GPUs, but those kernels
# need more shared memory than Ada cards have -> 'Shared memory size limit exceeded' at
# launch (which its trace-time fallback can't catch). Restrict Triton to true datacenter
# GPUs (A100 cc 8.0, H100 cc 9.0+); everything else uses XLA, exactly like the T4 path.
try:
  import tokamax
  _gu = os.path.join(os.path.dirname(tokamax.__file__), '_src', 'gpu_utils.py')
  _s = open(_gu).read()
  _old = 'return float(device.compute_capability) >= 8.0'
  _new = ('cc = float(device.compute_capability)\n'
          '  return cc == 8.0 or cc >= 9.0  # datacenter only; Ada/L4 (8.6/8.9) lack shared memory')
  if _old in _s:
    open(_gu, 'w').write(_s.replace(_old, _new))
    print('Patched tokamax: Triton restricted to datacenter GPUs (L4/Ada -> XLA).')
except Exception as _e:
  print(f'(tokamax patch skipped: {_e})')

# Weights, in the background. The ported models are fetched by the same code the run
# uses (alphafold3.model.weights.ensure_weights), so the run finds them already there
# and the cache layout cannot drift between the two. AlphaFold 3's own parameters are
# not ours to redistribute, so those come straight from Google.
STAMP = f'WEIGHTS_DONE_{model}_{PRECISION}'
if not os.path.isfile(STAMP):
  if IS_AF2:
    print('Downloading official AlphaFold 2 parameters (CC BY 4.0)...')
    with open('prefetch_af2.py', 'w') as fh:
      fh.write('import sys\n'
               'from alphafold3.model import weights\n'
               'print(weights.ensure_af2_params(sys.argv[1]))\n')
    os.system(f'(python prefetch_af2.py {AF2_DIR} > {STAMP}.log 2>&1 && touch {STAMP}) &')
  elif IS_AF3:
    print("Downloading official AlphaFold 3 weights (public, no login required)...")
    os.makedirs(NATIVE_DIR, exist_ok=True)
    for _f in glob.glob(f'{NATIVE_DIR}/*'):       # keep exactly one model file in the dir
      os.remove(_f)
    os.system(f'(wget -O {NATIVE_DIR}/af3.bin.zst "{AF3_WEIGHTS_URL}" > {STAMP}.log 2>&1 && touch {STAMP}) &')
  else:
    print(f'Downloading {model} weights...')
    with open('prefetch_weights.py', 'w') as fh:
      fh.write('import sys\n'
               'from alphafold3.model import weights\n'
               'print(weights.ensure_weights(sys.argv[1], None, precision=sys.argv[2]))\n')
    os.system(f'(python prefetch_weights.py {model} {PRECISION} > {STAMP}.log 2>&1 && touch {STAMP}) &')

# Where the compiled model is cached. /tmp is wiped when the VM goes away, so a
# fresh session recompiles (~53 s on a small input); Drive survives. Opt-in, and
# the run falls back to /tmp if the mount does not work rather than failing.
CACHE_DIR = '/tmp/af3_cache'
if persist_cache_to_drive:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    CACHE_DIR = '/content/drive/MyDrive/.af3_cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print(f'Compile cache: {CACHE_DIR} (survives this session)')
  except Exception as _e:
    print(f'(Drive mount failed, using {CACHE_DIR}: {_e})')

# Build AF3 data files (background, independent of weights)
if not os.path.isfile('DATA_DONE'):
  print('Fetching the CCD components this fold needs...')
  # NOT build_data. That parses libcifpp's whole components.cif -- 51,275
  # components, 518 MB -- into a 505 MB ccd.pickle, and costs 48 s of the
  # session. A fold references about thirty codes.
  #
  # LocalFold's trick: fetch each component from
  # files.rcsb.org/ligands/download/<CODE>.cif, kilobytes each, and build the
  # two pickles from just those. Measured: 0.6 s for 40 components, a 0.30 MB
  # pickle, and every field byte-identical to libcifpp's for ALA, SER, GOL,
  # ATP, SEP, NAG, DA and U.
  #
  # Both pickles come from the SAME fetched set, so they are self-consistent --
  # NAG lands in GLYCAN_LINKING_LIGANDS exactly as with the full dictionary. A
  # component the input names and we did not fetch raises KeyError, which is
  # loud, rather than being silently mis-bonded; hence the generous code list.
  # `ccd_fetch` ships in the wheel as of v3.1.8. It used to be fetched from
  # `main`, which meant the notebook mixed a tagged package with a moving
  # file -- the exact drift the pinned install above exists to prevent.
  with open('prefetch_ccd.py', 'w') as fh:
    fh.write(
      'import sys, os, importlib.metadata as md\n'
      'from alphafold3.constants import ccd_fetch\n'
      'root = os.path.dirname(md.distribution("alphafold3-colabfold")'
      '.locate_file("alphafold3"))\n'
      'conv = os.path.join(root, "alphafold3", "constants", "converters")\n'
      'os.makedirs(conv, exist_ok=True)\n'
      'ccd_fetch.write_pickles(ccd_fetch.codes_for_input(extra=sys.argv[1:]),\n'
      '  os.path.join(conv, "ccd.pickle"),\n'
      '  os.path.join(conv, "chemical_component_sets.pickle"))\n')
  os.system('(python prefetch_ccd.py > DATA_DONE.log 2>&1 && touch DATA_DONE) &')

# A BOUNDED wait. This used to be `while not exists: sleep(5)` with no limit
# and the background job's output discarded, so a failed download was an
# indefinite hang with nothing on screen -- which is exactly how it looked for
# ten minutes on 2026-09-17. Each job now writes <sentinel>.log, and a stall
# raises with the tail of it rather than waiting for the runtime to be
# reclaimed. The weights are the slow one: a few hundred MB, so minutes on a
# poor link, hence 20 of them before giving up.
def _await(sentinel, limit=1200):
  t0 = time.time()
  while not os.path.isfile(sentinel):
    if time.time() - t0 > limit:
      log = f'{sentinel}.log'
      tail = open(log).read()[-1500:] if os.path.isfile(log) else '(no output captured)'
      raise RuntimeError(f'{sentinel} did not appear within {limit} s. '
                         f'Tail of {log}:\n{tail}')
    time.sleep(5)
  print(f'{sentinel} ✓  ({time.time() - t0:.0f} s)')

for sentinel in (STAMP, 'DATA_DONE'):
  _await(sentinel)

if IS_AF3 and os.path.getsize(f'{NATIVE_DIR}/af3.bin.zst') < 1_000_000:
  raise RuntimeError('AlphaFold 3 weights download failed or incomplete - re-run this cell.')

print(f'Setup complete!  Model: {model}.')
if model == 'chai1':
  print('NOTE: chai-1 is running WITHOUT ESM2 embeddings, which are most of its token\n'
        '      features. Expect worse structures than chai-lab itself produces.')


In [ ]:
print('SETUP_SECONDS: %.0f' % (time.time()-T0))
seq='ACSEFGHIKLWYMNPQRSTV'
json.dump({'dialect':'alphafold3','version':4,'name':'t','modelSeeds':[1],
  'sequences':[{'protein':{'id':'A','sequence':seq,'unpairedMsa':'>q\n'+seq+'\n',
                           'pairedMsa':'','templates':[]}}]}, open('t.json','w'))
env=dict(os.environ, XLA_FLAGS='--xla_disable_hlo_passes=custom-kernel-fusion-rewriter')
t=time.time()
r=subprocess.run([sys.executable,'run_alphafold.py','--json_path=t.json',
  '--model='+model,'--output_dir=out','--norun_data_pipeline',
  '--num_diffusion_samples=1','--flash_attention_implementation=xla',
  '--buckets=32'],capture_output=True,text=True,env=env)
print('FOLD_SECONDS: %.0f rc=%d' % (time.time()-t, r.returncode))
if r.returncode: print((r.stdout+r.stderr)[-1200:])
c=glob.glob('out/**/*model.cif',recursive=True)
print('cifs:',len(c),'atoms:', sum(1 for l in open(c[0]) if l.startswith('ATOM')) if c else 0)
print('TOTAL_SECONDS: %.0f' % (time.time()-T0))
print('RESULT:', 'PASS' if (r.returncode==0 and c) else 'FAIL')
